# Introdução às Redes Neurais e Deep Learning

##  Redes Neurais Recorrentes (RNN)

### Introdução

As RNNs foram desenvolvidas especificamente para modelar dependências sequenciais mediante um estado recorrente, enquanto CNNs utilizam filtros para extrair padrões locais. As Redes Neurais Recorrentes permitem operar sobre sequências de entrada, saída ou ambas simultaneamente.
Alguns exemplos de utilização de uma rede recorrente são:
* Previsão de uma sequência temporal.
* Análise de sentimentos em textos.
* Geração de texto, caractere por caractere.


Os tipos de RNN podem ser:
* uma entrada, uma saída. Este é o tipo mais simples (trivial) de Redes Recorrentes, sem recorrência temporal efetiva. Ela é incluída aqui apenas para comparação com outras arquiteturas. 
* Uma entrada, Múltiplas saídas. neste tipo de RNN, a rede processa uma única entrada para produzir múltiplas saídas ao longo do tempo. Isso é útil em tarefas onde uma entrada desencadeia uma sequência de previsões (saídas). Por exemplo, na criação de legendas para imagens e geração de texto partir de um *token* (letra ou palavra).
* Muitas Entradas, Uma saída. Esse tipo é útil quando o contexto geral da sequência de entrada é necessário para fazer uma única previsão. Na análise de sentimentos, o modelo recebe uma sequência de palavras (como uma frase) e produz uma única saída, como positiva, negativa ou neutra.
* Muitas Entradas, Muitas Saídas. Esta rede Processa uma sequência de entradas e gera uma sequência de saídas. Na tarefa de tradução de idiomas, uma sequência de palavras em um idioma é fornecida como entrada e uma sequência correspondente em outro idioma é gerada como saída.
<div>
<img src="Files/Tipos de RNNs_ Entradas e Saídas.png" width="500" height="500">   
</div>  
Fonte: Gerado com chatGPT
  

Basicamente, uma RNN é uma caixa preta, que possui um "estado interno" o qual é atualizado à medida que uma sequência é processada. A cada passo de tempo, a rede pode ser considerada como uma rede *feedforward* padrão , ou seja, densamente conectada, mas com uma conexão adicional vinda do estado oculto anterior. Desta forma, esta rede modifica seu estado em função suas entradas e dos estados anteriores.
<div>
<img src="Files/RNN_Diag_GPT.png" width="500" height="500">   
</div>

  Fonte: Gerado com ChatGPT.


 O aprendizado é realizado com a  retropropagação no tempo (BPTT), um processo  no qual os erros são propagados ao longo dos intervalos de tempo para ajustar os pesos da rede, aprimorando a capacidade da RNN de aprender dependências em dados sequenciais. Ao ajustarmos os pesos da RNN, ela apresentará comportamentos diferentes em termos de como seu estado evolui conforme recebe essas entradas. É importante frisar que o conjunto de pesos é o mesmo para todos os instantes de tempo e estados associados. 

As RNNs compartilham semelhanças nas estruturas de entrada e saída com outras arquiteturas de redes neurais, mas diferem significativamente na forma como a informação flui da entrada para a saída. Ao contrário das redes neurais profundas tradicionais, onde cada camada densa possui matrizes de pesos distintas, as RNNs utilizam pesos compartilhados ao longo do tempo, permitindo que elas retenham informações sobre as sequências.

Em RNNs, o estado interno $h_t$ é o estado no tempo corrente e é calculado para cada entrada $x_t$, retendo dependências sequenciais. 
            
A cada passo de tempo, as RNNs processam unidades com uma função de ativação fixa. O estado oculto interno atua como memória, retendo informações de passos de tempo anteriores e essa memória permite que a rede armazene conhecimento passado e se adapte com base em novas entradas.

$$h_t=f_a(\underbrace{w_{hh}h_{t-1}+w_{xh}x_t+b_h}_{z_t})$$
onde $f_a$ é a função de ativação, normalmente $\tanh$, $w_{hh}$ e $w_{xh}$ são os pesos, $b_h$ o viés (*bias*) e $h_{t-1}$ é o estado anterior, e para a saída:

$$\hat{y}=f_y(o)=f_y(\underbrace{w_{hy}h_n+b_y}_{o})$$

onde **$h_n$ é o estado oculto final** fluindo para a saída da rede e $f_y(o)$ é a função de ativação na saída, usualmente a função softmax. 

### Retropropagação através do Tempo (BPTT)
Para simplificar o desenvolvimento matemático da BPTT, consideraremos inicialmente uma RNN com uma única unidade recorrente, de modo que as entradas, os estados ocultos e os pesos sejam escalares. Posteriormente, os resultados poderão ser generalizados para vetores e matrizes.

A função de Perdas $L$ depende do estado final e dos estados anteriores e **esses influenciam o cálculo de $L$ indiretamente, como veremos**. Dizendo de outa maneira, $h_n$ (último estado oculto) depende de $h_{n-1}$, que depende de $h_{n-2}$ e  sussessivamente até  $h_0$.  Assim,  na BPTT, os gradientes são calculados através de cada passo no tempo. Inicialmente, usando a regra da cadeia são calculados os gradientes para o estado final $h_n$:

$$\frac{\partial L}{\partial w_{hy}}=\frac{\partial L}{\partial o}\frac{\partial o}{\partial w_{hy}}$$

Logo, 
$$\frac{\partial o}{\partial w_{hy}}=h_n,$$

$$\frac{\partial L}{\partial w_{hy}}=\frac{\partial L}{\partial o}h_n.$$

Além disso, $\frac{\partial o}{\partial b_y}=1$, e $\frac{\partial L}{\partial b_y}=\frac{\partial L}{\partial o}$.

Supondo que a função de ativação  na saída é a função sigmoide e a função de perdas é a entropia cruzada binária , teremos como visto anteriormente:

 $$\frac{\partial L}{\partial o}=\hat y_i-y_i$$

Na sequência, determinamos os gradientes relativamente a $w_{hh},w_{xh}\;\text{e}\;b_h$ e para tanto teremos que fazer a retropropagação através de todos os passos de tempo, ou seja BPTT. 

Primeiro, calculamos
$$\frac{\partial L}{\partial h_n}=\frac{\partial L}{\partial o}\frac{\partial o}{\partial h_n}=\frac{\partial L}{\partial o}w_{hy}$$
e então
$$\frac{\partial L}{\partial z_n}=\frac{\partial L}{\partial o}\frac{\partial o}{\partial h_n}\frac{\partial h_n}{\partial z_n}$$
 
 Por outro lado, em um instante $t$ dado $h_t=\tanh(\underbrace{w_{xh}x_t+w_{hh}h_{t-1}+b_h}_{z_t})$, e a derivada da tangente hiperbólica é $\frac{\partial \,\tanh(x)}{\partial x}=1-\tanh^2(x)$. Portanto temos que
 $$\boxed{\frac{\partial L}{\partial z_n}=\frac{\partial L}{\partial o}w_{hy}(1-h_n^2)}$$

Assim, para todos os estados anteriores temos

$$\boxed{\frac{\partial L}{\partial z_{i}}=\frac{\partial L}{\partial z_n}\prod_{t=n}^{i+1} \left[\frac{\partial z_t}{\partial h_{t-1}}\frac{\partial h_{t-1}}{\partial z_{t-1}}\right]}$$
com o produtório percorrido ordem decrescente. Para um implementação escalar temos ($\frac{\partial z_t}{\partial h_{t-1}}=w_{hh})$:
$$\boxed{
\frac{\partial L}{\partial z_{i}}=\frac{\partial L}{\partial z_n}\prod_{t=n}^{i+1}w_{hh}(1-h_{t-1}^2)}$$

Como os parâmetros $w_{hh},\;w_{xh}\;\text{e}\;b$ são usados em todos os instantes de tempo, suas contribuições são somadas, isto é, 
$$\frac{\partial L}{\partial w_{hh}}=\sum_{i=1}^n\frac{\partial L}{\partial z_i}\frac{\partial z_i}{\partial w_{hh}}=\sum_{i=1}^n\frac{\partial L}{\partial z_i}h_{i-1}$$
$$\frac{\partial L}{\partial w_{xh}}=\sum_{i=1}^n\frac{\partial L}{\partial z_i}\frac{\partial z_i}{\partial w_{xh}}=\sum_{i=1}^n\frac{\partial L}{\partial z_i}x_{i}$$

$$\frac{\partial L}{\partial b_h}=\sum_{i=1}^n\frac{\partial L}{\partial z_i}\frac{\partial z_i}{\partial b_h}=\sum_{i=1}^n\frac{\partial L}{\partial z_i}$$

Nas últimas 3 expressões, a dependência indireta dos estados $h_t$ em relação ao aos estados anteriores já foi incorporada nos cálculos das derivadas $\frac{\partial L}{\partial z_{i}}$. 




In [1]:
from tqdm import tqdm
import numpy as np

##### Data #####
train_X = ['bom', 'bem', 'ruim', 'feliz', 'triste', 'não bom', 'não ruim',
       'não feliz', 'não triste', 'muito bom', 'muito ruim',
       'muito feliz', 'muito triste', 'eu estou feliz', 'isto é bom',
       'eu estou ruim', 'isto é ruim', 'eu estou triste', 'isto é triste',
       'eu não estou feliz', 'isto não é bom', 'eu não estou ruim',
       'isto não é triste', 'eu estou muito feliz', 'isto é muito bom',
       'eu estou muito ruim', 'isto é muito triste', 'isto é muito feliz',
       'eu estou bem não ruim', 'isto é bom não ruim',
       'eu estou ruim não bem', 'eu estou bem e feliz',
       'isto não é bom e não é feliz', 'eu não estou nada bem',
       'eu não estou nada ruim', 'eu não estou nada feliz',
       'isto não é nada triste', 'isto não é nada feliz',
       'eu estou bem agora', 'eu estou ruim agora',
       'isto está ruim agora', 'eu estou triste agora',
       'eu estava bem antes', 'eu estava feliz antes',
       'eu estava ruim antes', 'eu estava triste antes',
       'eu estou muito ruim agora', 'isto está muito bom agora',
       'isto está muito triste agora', 'isto estava ruim antes',
       'isto estava muito bom antes', 'isto estava muito ruim antes',
       'isto estava muito feliz antes', 'isto estava muito triste antes',
       'eu estava bem e não ruim antes',
       'eu não estava bem e não estava feliz antes',
       'eu não estou nada ruim ou triste agora',
       'eu não estou nada bem ou feliz agora',
       'isto não estava feliz e nem bom antes',
       'isto não é feliz',
       ]
train_y = [1, 1,0, 1, 0, 0, 1, 0, 1, 1, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 1, 
           1, 0, 0, 1, 1, 1, 0, 1, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 
           1, 0, 0, 1, 0, 1, 0, 1, 0, 1, 0, 0,0]

test_X = [
    'isto é feliz',
    'eu estou bem',
    'eu não estou bem',
    'isto não é ruim',
    'eu não estou triste',
    'eu estou muito bem',
    'isto é muito ruim',
    'eu estou muito triste',
    'isto é ruim não bom',
    'isto é bom e feliz',
    'eu não estou bem e não estou feliz',
    'eu não estou nada triste',
    'isto não é nada bom',
    'isto não é nada ruim',
    'isto está bom agora',
    'isto está triste agora',
    'isto está muito ruim agora',
    'isto estava bom antes',
    'eu não estava feliz e não estava bem antes'
]
test_y = [1, 1, 0, 1, 1, 1, 0, 0, 0, 1, 0, 1, 0, 1, 1, 0, 0, 1, 0]

In [2]:
vocab = set([q for text in train_X for q in text.split()])
vocab_size = len(vocab)

word_to_index = {w:i for i, w in enumerate(vocab)}
# transforma texto em índices
def text_to_sequence(text):
    return [word_to_index[word] for word in text.split()]

X_train_seq = [text_to_sequence(text) for text in train_X]
X_test_seq  = [text_to_sequence(text) for text in test_X]
max_len = max(len(seq) for seq in X_train_seq)

In [4]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense,Input
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical

In [7]:
X_train_pad = pad_sequences(X_train_seq, maxlen=max_len, padding='post')
X_test_pad  = pad_sequences(X_test_seq, maxlen=max_len, padding='post')

X_train_oh = to_categorical(X_train_pad, num_classes=vocab_size)
X_test_oh  = to_categorical(X_test_pad, num_classes=vocab_size)

In [8]:
model = Sequential([
    Input((max_len,vocab_size)),
    SimpleRNN(64,activation='tanh'),
    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ simple_rnn_1 (SimpleRNN)             │ (None, 64)                  │           5,376 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 1)                   │              65 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 5,441 (21.25 KB)

 Trainable params: 5,441 (21.25 KB)

 Non-trainable params: 0 (0.00 B)

In [9]:
y_train = np.array(train_y)
y_test  = np.array(test_y)

history = model.fit(
    X_train_oh,
    y_train,
    epochs=100,
    batch_size=4,
    validation_data=(X_test_oh, y_test),verbose=0
)

loss, acc = model.evaluate(X_test_oh, y_test)

print("\nAccuracy:", acc)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step - accuracy: 1.0000 - loss: 0.0026

Accuracy: 1.0


In [10]:
accuracy = 0
for input, label in zip(test_X, y_test):
    
    print(input)

    seq = text_to_sequence(input)
    seq = pad_sequences([seq], maxlen=max_len, padding='post')
    seq = to_categorical(seq, num_classes=vocab_size)

    prediction = model.predict(seq,verbose='False')
    if prediction > 0.5:
        print("positivo")
        pred=1
    else:
        print("negativo")
        pred=0
    if pred == label:
        accuracy += 1

print(f'\nAccuracy: {round(accuracy * 100 / len(test_X), 2)}%')

isto é feliz
positivo
eu estou bem
positivo
eu não estou bem
negativo
isto não é ruim
positivo
eu não estou triste
positivo
eu estou muito bem
positivo
isto é muito ruim
negativo
eu estou muito triste
negativo
isto é ruim não bom
negativo
isto é bom e feliz
positivo
eu não estou bem e não estou feliz
negativo
eu não estou nada triste
positivo
isto não é nada bom
negativo
isto não é nada ruim
positivo
isto está bom agora
positivo
isto está triste agora
negativo
isto está muito ruim agora
negativo
isto estava bom antes
positivo
eu não estava feliz e não estava bem antes
negativo

Accuracy: 100.0%


In [31]:
# Definição do Texto de Entrada e Identificação do Vocabulário
text = "These are the voyages of the starship Enterprise Its five year mission "\
        "To explore strange new worlds To seek out new life "
chars = sorted(list(set(text)))
char_to_index = {char: i for i, char in enumerate(chars)}
index_to_char = {i: char for i, char in enumerate(chars)}

In [32]:
# Criação de Sequências de Comprimento Fixo e dos Rótulos
seq_length = 3
sequences = []
labels = []

for i in range(len(text) - seq_length):
    seq = text[i:i + seq_length]
    label = text[i + seq_length]
    sequences.append([char_to_index[char] for char in seq])
    labels.append(char_to_index[label])

X = np.array(sequences)
y = np.array(labels)

In [33]:
# Codificação da Entrada e do Rótulos em vetores One-Hot:
X_one_hot = tf.one_hot(X, len(chars))
y_one_hot = tf.one_hot(y, len(chars))

In [34]:
# Modelo RNN
# batch_size FIXO
batch_size = 1

# Modelo stateful
model = Sequential([
    Input(batch_shape=(1, seq_length, len(chars))),

    SimpleRNN(
        64,
        activation='tanh',
        stateful=True
    ),

    Dense(len(chars), activation='softmax')
])
model.summary()
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

Model: "sequential_8"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ simple_rnn_8 (SimpleRNN)             │ (1, 64)                     │           5,760 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_8 (Dense)                      │ (1, 25)                     │           1,625 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 7,385 (28.85 KB)

 Trainable params: 7,385 (28.85 KB)

 Non-trainable params: 0 (0.00 B)

In [35]:
# Treinamento
epochs = 100

for epoch in range(epochs):

    # reset no início da época
    model.layers[0].reset_states()

    model.fit(
        X_one_hot,
        y_one_hot,
        epochs=1,
        batch_size=batch_size,
        shuffle=False,
        verbose=True
    )

119/119 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.0175 - loss: 3.2166    
119/119 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.2761 - loss: 2.6770 
119/119 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.2948 - loss: 2.4974 
119/119 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.3082 - loss: 2.3127 
119/119 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.3506 - loss: 2.1472
119/119 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.3864 - loss: 1.9932
119/119 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.4865 - loss: 1.8324
119/119 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.4997 - loss: 1.6793
119/119 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5432 - loss: 1.5335
119/119 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5625 - loss: 1.3998
119/119 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5965 - loss: 1.2990
119/119 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.6957 - loss: 1.1997
119/119 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.6799 - loss: 1.1620
119/119 ━━━━━━━━━━

In [36]:
# =========================
# Geração de texto
# =========================

# reset antes da inferência
model.layers[0].reset_states()

start_seq = "The"
generated_text = start_seq

for i in range(len(text)):
    x = np.array([[char_to_index[char] for char in generated_text[-seq_length:]]])
    x_one_hot = tf.one_hot(x, len(chars))
    prediction = model.predict(
       x_one_hot,
       batch_size=batch_size,
       verbose=0
   )
    next_index = np.argmax(prediction)
    next_char = index_to_char[next_index]
    generated_text += next_char

print("Generated Text:")
print(generated_text)

Generated Text:
These are the voyages of the starship Enterprise Its five year mission To explore strange new worlds To seek out new life see
